# Data Cleaning — Match

In [32]:
import pandas as pd

# Keep ALL matches — away results needed for form engineering
df = pd.read_csv('../../data/raw/gold_match.csv', sep=',', on_bad_lines='skip')
print(df.shape)
df.head()

(142, 27)


,match_id,match_date,kickoff_time_local,match_date_utc,kickoff_time_utc,matchday,competition_id,competition_name,season,season_id,...,venue,winner,result_home,goals_home_ht,goals_away_ht,goals_home_ft,goals_away_ft,is_home_match,last_result_vs_opponent,tickets_scanned
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022-07-23Z,16:15:00Z,1.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Guldensporenstadion,away,L,0,0,0,2,False,L 1-2,NaN
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022-07-30Z,16:15:00Z,2.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,home,W,1,0,2,0,True,NaN,5565.0
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,2022-08-07Z,19:00:00Z,3.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Bosuilstadion,home,W,2,1,4,2,False,L 0-1,NaN
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,2022-08-14Z,16:30:00Z,4.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,away,L,0,2,0,3,True,L 1-4,7440.0
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,2022-08-21Z,16:30:00Z,5.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Stade Maurice Dufrasne,away,L,0,2,1,3,False,W 2-1,NaN


In [33]:
# Fix data types
df['match_date'] = pd.to_datetime(df['match_date'])
df['kickoff_time_local'] = pd.to_datetime(df['kickoff_time_local'], format='%H:%M:%S').dt.time
# Pandas reads true/false from CSV as Python booleans — map with string keys would silently fail
df['is_home_match'] = df['is_home_match'].astype(bool)
df['matchday'] = df['matchday'].astype('Int64')

In [34]:
# Duplicate check
print(f"Duplicate match_id rows: {df['match_id'].duplicated().sum()}")

# Missing value overview
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nNote: tickets_scanned is null for away matches ({(~df['is_home_match']).sum()} rows) — expected")

Duplicate match_id rows: 0

Missing values:
last_result_vs_opponent     4
tickets_scanned            62
dtype: int64

Note: tickets_scanned is null for away matches (71 rows) — expected


In [35]:
# Must be done BEFORE dropping home_team / away_team

# Opponent: away_team when OHL is home, home_team when OHL is away
df['opponent'] = df.apply(
    lambda r: r['away_team'] if r['is_home_match'] else r['home_team'], axis=1
)

# OHL result: result_home is from home team perspective — flip for away games
def ohl_result(row):
    if row['is_home_match']:
        return row['result_home']
    flip = {'W': 'L', 'L': 'W', 'D': 'D'}
    return flip.get(row['result_home'], None)

df['ohl_result'] = df.apply(ohl_result, axis=1)

# Goals from OHL's perspective
df['ohl_goals_ft']      = df.apply(lambda r: r['goals_home_ft'] if r['is_home_match'] else r['goals_away_ft'], axis=1)
df['opponent_goals_ft'] = df.apply(lambda r: r['goals_away_ft'] if r['is_home_match'] else r['goals_home_ft'], axis=1)

In [36]:
# Parse last_result_vs_opponent (e.g. "L 1-2") into result letter and integer goal columns
# Format is from OHL's perspective: result + ohl_goals-opponent_goals
df['last_h2h_result']        = df['last_result_vs_opponent'].str.extract(r'^([WDL])')
df['last_h2h_ohl_goals']     = df['last_result_vs_opponent'].str.extract(r'(\d+)-\d+$').astype('Int64')
df['last_h2h_opponent_goals']= df['last_result_vs_opponent'].str.extract(r'\d+-(\d+)$').astype('Int64')
df = df.drop(columns=['last_result_vs_opponent'])

In [37]:
# Drop redundant, constant, and internal columns
cols_to_drop = [
    'match_date_utc',     # redundant with match_date
    'kickoff_time_utc',   # redundant with kickoff_time_local
    'competition_id',     # internal ID
    'competition_name',   # constant
    'season_id',          # internal ID
    'home_team_id',       # internal ID
    'away_team_id',       # internal ID
    'home_team_code',     # redundant with opponent
    'away_team_code',     # redundant with opponent
    'home_team',          # replaced by opponent column
    'away_team',          # replaced by opponent column
    'venue',              # constant for home games, irrelevant for away
    'winner',             # replaced by ohl_result
    'result_home',        # replaced by ohl_result
    'goals_home_ft',      # replaced by ohl_goals_ft / opponent_goals_ft
    'goals_away_ft',      # replaced by ohl_goals_ft / opponent_goals_ft
    'goals_home_ht',      # half-time scores — minimal attendance prediction value
    'goals_away_ht',
]
df = df.drop(columns=cols_to_drop)

print(df.shape)
print(df.dtypes)
df.head(15)

(142, 15)
match_id                              str
match_date                 datetime64[us]
kickoff_time_local                 object
matchday                            Int64
season                                str
stage                                 str
is_home_match                        bool
tickets_scanned                   float64
opponent                              str
ohl_result                            str
ohl_goals_ft                        int64
opponent_goals_ft                   int64
last_h2h_result                       str
last_h2h_ohl_goals                  Int64
last_h2h_opponent_goals             Int64
dtype: object


,match_id,match_date,kickoff_time_local,matchday,season,stage,is_home_match,tickets_scanned,opponent,ohl_result,ohl_goals_ft,opponent_goals_ft,last_h2h_result,last_h2h_ohl_goals,last_h2h_opponent_goals
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,1,2022/2023,Regular Season,False,NaN,Kortrijk,W,2,0,L,1,2
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2,2022/2023,Regular Season,True,5565.0,Westerlo,W,2,0,NaN,<NA>,<NA>
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,3,2022/2023,Regular Season,False,NaN,Antwerp,L,2,4,L,0,1
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,4,2022/2023,Regular Season,True,7440.0,Club Brugge,L,0,3,L,1,4
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,5,2022/2023,Regular Season,False,NaN,Standard Liège,W,3,1,W,2,1
5,d65hmi7sq03yzr5he1k7ypus4,2022-08-27,18:15:00,6,2022/2023,Regular Season,True,4489.0,KV Oostende,W,2,1,W,3,1
6,d700mwu32x772ns4veqq32yhg,2022-09-04,13:30:00,7,2022/2023,Regular Season,False,NaN,Anderlecht,D,2,2,D,0,0
7,d80mkemezkz16bqh6lbn8tlhw,2022-09-10,20:45:00,8,2022/2023,Regular Season,True,4508.0,Sporting Charleroi,W,3,2,W,3,0
8,d9pbl588401et3engzzakcfmc,2022-09-17,16:00:00,9,2022/2023,Regular Season,False,NaN,Mechelen,D,0,0,L,0,2
9,dak40etbhbqsr1nxyt50qcg0k,2022-10-01,16:00:00,10,2022/2023,Regular Season,True,6290.0,Union Saint-Gilloise,L,0,3,L,1,4


In [38]:
df.to_csv('../../data/cleaned/gold_match_cleaned.csv', index=False)
print('Saved.')

Saved.
